In [0]:
# 1 - Parâmetros e ajustes do Notebook:

"""Ajustando o caminho do Volume onde os CSVs foram carregados"""

dbutils.widgets.text(
    "volume_path",
    "/Volumes/workspace/olist_ecommerce/raw_data",
    "Caminho do Volume (Unity Catalog)"
)

"""O Dataset Olist Ecommerce cobre uma faixa de pedidos que abrangem de setembro de 2016 até agosto de 2018,
então, basicamente, estamos setando a data de início do período de cotação PTAX, ajustando para o padrão americano de data (MM-DD-AAAA)"""

# Data inicial do período de cotação
dbutils.widgets.text(
    "data_inicio",
    "01-01-2016",
    "Data Início PTAX (MM-DD-AAAA)"
)

# Data final do período de cotação
dbutils.widgets.text(
    "data_fim",
    "12-31-2018",
    "Data Fim PTAX (MM-DD-AAAA)"
)

"""Leitura dos widgets definidos anteriormente"""

VOLUME_PATH = dbutils.widgets.get("volume_path")
DATA_INICIO = dbutils.widgets.get("data_inicio")
DATA_FIM    = dbutils.widgets.get("data_fim")

print("Parâmetros ativos nesta execução:")
print(f"Volume Path : {VOLUME_PATH}")
print(f"PTAX de     : {DATA_INICIO}")
print(f"PTAX até    : {DATA_FIM}")

Parâmetros ativos nesta execução:
Volume Path : /Volumes/workspace/olist_ecommerce/raw_data
PTAX de     : 01-01-2016
PTAX até    : 12-31-2018


In [0]:
# 2 - Criação do banco de dados da camada Bronze:

"""
Criação do Banco de Dados da camada Bronze:

- Usamos o IF NOT EXISTS para tornar a operação idempotente, ou seja, se o schema já existir, ele não será criado novamente.
- Definimos bronze como schema padrão desta sessão Spark.
- Sem isso, cada referência de tabela precisaria ser prefixada com 'bronze'. (ex: bronze.tb_orders em vez de apenas tb_orders).
- O COMMENT serve para descrever o propósito do schema.
"""

spark.sql("""
    CREATE DATABASE IF NOT EXISTS bronze
    COMMENT 'Camada Bronze — Dados brutos ingeridos sem transformação'
""")

# Define bronze como schema padrão desta sessão
spark.sql("USE bronze")

# Confirmação: lista todos os databases para validar a criação
print("Banco de dados 'bronze' criado.")
print("Databases disponíveis no ambiente:")
spark.sql("SHOW DATABASES").show(truncate=False)


Banco de dados 'bronze' criado/verificado.

Databases disponíveis no ambiente:
+------------------+
|databaseName      |
+------------------+
|bronze            |
|default           |
|information_schema|
|olist_ecommerce   |
+------------------+



In [0]:
# 3 - Ingestão dos CSVs:

# 3.1 - Imports e mapeamento dos arquivos:

"""
Imports:
Importamos a biblioteca pyspark.sql.functions (alias F) para usar algumas funções nativas do Spark.
- A função F.current_timestamp() retorna o timestamp do momento de execução.
- As funções do módulo F rodam no cluster (workers), não no driver, tornando-se mais eficientes que funções Python puras em UDFs

Mapeamento:
- Dicionário {nome_arquivo_csv: nome_tabela_bronze}
- Chave : Nome exato do arquivo no Volume
- Valor : Nome da tabela a ser criada no Schema bronze
"""

from pyspark.sql import functions as F

MAPEAMENTO = {
    "olist_customers_dataset.csv"           : "tb_customers",
    "olist_geolocation_dataset.csv"         : "tb_geolocalizacao",
    "olist_order_items_dataset.csv"         : "tb_order_items",
    "olist_order_payments_dataset.csv"      : "tb_order_payments",
    "olist_order_reviews_dataset.csv"       : "tb_order_reviews",
    "olist_orders_dataset.csv"              : "tb_orders",
    "olist_products_dataset.csv"            : "tb_products",
    "olist_sellers_dataset.csv"             : "tb_sellers",
    "product_category_name_translation.csv": "tb_product_category_name_translation",
}

print(f"Arquivos mapeados para ingestão: {len(MAPEAMENTO)}")
print()
for arquivo, tabela in MAPEAMENTO.items():
    print(f"  {arquivo:<50} → bronze.{tabela}")


Arquivos mapeados para ingestão: 9

  olist_customers_dataset.csv                        → bronze.tb_customers
  olist_geolocation_dataset.csv                      → bronze.tb_geolocalizacao
  olist_order_items_dataset.csv                      → bronze.tb_order_items
  olist_order_payments_dataset.csv                   → bronze.tb_order_payments
  olist_order_reviews_dataset.csv                    → bronze.tb_order_reviews
  olist_orders_dataset.csv                           → bronze.tb_orders
  olist_products_dataset.csv                         → bronze.tb_products
  olist_sellers_dataset.csv                          → bronze.tb_sellers
  product_category_name_translation.csv              → bronze.tb_product_category_name_translation


In [0]:
# 3.2 - Função generalizada para a ingestão dos CSVs:

"""
Função de ingestão principal (generalizada para poder ingerir qualquer CSV, evitando repetição de código desnecessária)

Args:
    nome_arquivo : Nome do arquivo CSV (ex: 'olist_orders_dataset.csv')
    nome_tabela  : Nome da tabela Bronze de destino (ex: 'tb_orders')

Returns:
    int: Quantidade de registros ingeridos

Raises:
    Exception: Propaga qualquer erro de leitura ou escrita para o chamador

Funcionamento geral:

1. Lê o CSV do Volume usando Spark:

spark.read.option(...).csv(caminho)
. header=True: usa a 1ª linha como nomes de coluna.
. inferSchema=True: O Spark faz 2 passagens no arquivo - A 1° para detectar tipos e a 2° para ler os dados.
É um pouco mais lento, mas elimina a necessidade de declarar o schema manualmente na Bronze.
. encoding=UTF-8: Tratamento para cidades e nomes de produtos brasileiros que possuem acentos.

2. Adiciona uma coluna com o Timestamp da ingestão:

-> withColumn('timestamp_ingestion', F.current_timestamp()): Adiciona uma nova coluna com o instante exato da execução.

. current_timestamp() é avaliado no momento da escrita no Delta, não da leitura.

Isso significa que cada tabela terá o timestamp real de quando seus dados foram persistidos no storage.

3. Grava o DataFrame como tabela Delta no Unity Catalog:

-> write.format('delta').mode('overwrite').option('overwriteSchema','true')

. format('delta'): Delta Lake oferece transações ACID, time travel e schema enforcement no formato Delta.
. mode('overwrite'): Substitui tudo a cada execução, garantindo que reexecuções não dupliquem dados.
. overwriteSchema='true': Permite alterar o Schema entre execuções. Isso é necessário, pois adicionar/remover uma coluna poderia causar um erro e não permitir a ingestão de dados.
. saveAsTable(...): registra a tabela no metastore (Unity Catalog). Diferente de .save() que apenas grava arquivos no storage sem registro no Catalog.

4. Retorna a contagem de linhas lidas

"""


def ingerir_csv(nome_arquivo: str, nome_tabela: str) -> int:

    caminho = f"{VOLUME_PATH}/{nome_arquivo}"

    # Leitura direta do CSV, sem nenhuma transformação:
    df = (
        spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .option("encoding", "UTF-8")
            .csv(caminho)
            .withColumn("timestamp_ingestion", F.current_timestamp())
    )

    # Acionando a execução real:
    total = df.count()

    # Gravação como tabela Delta registrada no Unity Catalog:
    (
        df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(f"bronze.{nome_tabela}")
    )

    return total

print("Função ingerir_csv() definida e pronta para uso.")

Função ingerir_csv() definida e pronta para uso.


In [0]:
# 3.3 - Execução da Iteração de Ingestão:

"""
Descrição: 

Loop: 
- A iteração ocorre sobre o dicionário MAPEAMENTO e chamamos a função ingerir_csv() para cada par.

Tratamento de erros com try/except:
- Cada arquivo é processado independentemente.
- Se um arquivo falhar (por exemplo: CSV corrompido, caminho errado, permissão), o erro é capturado e registrado, mas o loop continua para os demais. O que permite o processamento dos outros arquivos sem interromper o fluxo.

Por fim, a lista 'resultados' acumula (tabela, contagem, status) para o resumo.
"""

resultados = []

print("Iniciando ingestão dos CSVs...")

for arquivo, tabela in MAPEAMENTO.items():
    try:
        total = ingerir_csv(arquivo, tabela)
        resultados.append((tabela, total, "OK"))
        print(f"[OK] bronze.{tabela:<43} {total:>8,} registros")

    except Exception as e:
        resultados.append((tabela, 0, f"ERRO"))
        print(f"[Erro] bronze.{tabela}: {str(e)[:70]}")


# Resumo consolidado do processamento:
ok = sum(1 for _, _, s in resultados if s == "OK")
erros = sum(1 for _, _, s in resultados if s != "OK")
total_registros = sum(t for _, t, s in resultados if s == "OK")

print(f"\nResumo da ingestão:")
print(f"Tabelas carregadas: {ok}/{len(MAPEAMENTO)}")
print(f"Total de registros: {total_registros:,}")

if erros:
    print(f"Tabelas com erro: {erros} — Verificar o caminho do Volume e os nomes dos arquivos")

Iniciando ingestão dos CSVs...
[OK] bronze.tb_customers                                  99,441 registros
[OK] bronze.tb_geolocalizacao                           1,000,163 registros
[OK] bronze.tb_order_items                               112,650 registros
[OK] bronze.tb_order_payments                            103,886 registros
[OK] bronze.tb_order_reviews                             104,162 registros
[OK] bronze.tb_orders                                     99,441 registros
[OK] bronze.tb_products                                   32,951 registros
[OK] bronze.tb_sellers                                     3,095 registros
[OK] bronze.tb_product_category_name_translation              71 registros

Resumo da ingestão:
Tabelas carregadas: 9/9
Total de registros: 1,555,860


In [0]:
# 4 - Ingestão da API PTAX - Cotação do dólar:

# 4.1 - Teste da API:
"""
Antes de ingerir os dados, vamos testar se a API está acessível neste ambiente.

- Se estiver, vamos fazer uma requisição para a lista de moedas disponíveis e exibir o resultado.
- Se não estiver, vamos fazer uma requisição para o CSV local e exibir o resultado.

A definição de timeout=10:

- Evita que o notebook trave indefinidamente se o domínio estiver bloqueado por firewall ou regra de egress.- Após 10 segundos sem resposta, lança requests.exceptions.Timeout.

A variável API_ACESSIVEL controla quais células são executadas a seguir:

- True  → Seção 4B (ingestão via API)
- False → Seção 4C (fallback via CSV local)
"""

import requests

URL_TESTE = "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"

try:
    resp_teste = requests.get(URL_TESTE, timeout=10)
    API_ACESSIVEL = resp_teste.status_code == 200
    print(f"API BCB: Acessível (HTTP {resp_teste.status_code}) — prosseguindo com ingestão online.")
except Exception as e:
    API_ACESSIVEL = False
    print(f"API BCB: Inacessível — {type(e).__name__}: {e}")

API BCB: Acessível (HTTP 200) — prosseguindo com ingestão online.


In [0]:
# 4.2 - Realização da ingestão via API PTAX:

"""
Descrição:

Endpoint: CotacaoDolarPeriodo
- Parâmetros: dataInicial e dataFinalCotacao no formato MM-DD-AAAA
- Retorno: cotações diárias (compra) do dólar para um intervalo de datas
- select: limita os campos retornados
- format: resposta em JSON (alternativa: XML)

Tratamentos:
- raise_for_status(): Lança HTTPError automaticamente se o status HTTP for 4xx ou 5xx
- get('value', []): O JSON retornado pela API OData tem a estrutura: { "@odata.context": "...", "value": [ {...}, {...} ] }
- O default [] evita KeyError se a API retornar resposta inesperada
- spark.createDataFrame(df_pandas): Converte Pandas DataFrame para Spark DataFrame
"""

import pandas as pd

if API_ACESSIVEL:

    URL_PTAX = (
        "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
        f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
        f"?@dataInicial='{DATA_INICIO}'"
        f"&@dataFinalCotacao='{DATA_FIM}'"
        f"&$select=dataHoraCotacao,cotacaoCompra"
        f"&$format=json"
    )

    print("Consultando API PTAX...")
    print(f"Período: {DATA_INICIO} → {DATA_FIM}")

    # Requisição HTTP com timeout de 30s para o payload completo
    resposta = requests.get(URL_PTAX, timeout=30)
    resposta.raise_for_status()

    # Extraindo a lista de cotações do campo 'value'
    lista_cotacoes = resposta.json().get("value", [])

    if not lista_cotacoes:
        raise ValueError("API retornou 0 cotações. Verifique o intervalo de datas.")

    print(f"Cotações recebidas: {len(lista_cotacoes)} registros")

    # Pandas -> Spark -> Delta
    df_ptax = (
        spark
            .createDataFrame(pd.DataFrame(lista_cotacoes))
            .withColumn("timestamp_ingestion", F.current_timestamp())
    )

    (
        df_ptax.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable("bronze.tb_cotacao_dolar")
    )

    print(f"bronze.tb_cotacao_dolar salva com {len(lista_cotacoes):,} registros.")

else:
    print("[PROBLEMA] API inacessível.")

Consultando API PTAX...
Período: 01-01-2016 → 12-31-2018
Cotações recebidas: 750 registros
bronze.tb_cotacao_dolar salva com 750 registros.


In [0]:
# 5 - Verificação e Validação do processamento:

"""
Basicamente, aqui vamos fazer um check básico para ver se todas as tabelas foram criadas corretamente.

SHOW TABLES IN bronze:
- Executa uma query SQL no Spark SQL.
- A query retorna um DataFrame com 2 colunas: tableName e isTemporary.
Funções:
- A função .count() retorna o número de linhas no DataFrame.
- A função display() renderiza o DataFrame como uma tabela interativa no notebook.
"""
# 5.1 - Listagem das tabelas no Schema 'bronze':

df_tabelas = spark.sql("SHOW TABLES IN bronze")
print("Tabelas registradas no schema 'bronze':")
display(df_tabelas)

total_tabelas = df_tabelas.count()
status = "OK" if total_tabelas == 10 else "VERIFICAR"
print(f"\nTotal encontrado: {total_tabelas}/10 tabelas [{status}]")

# 5.2 Validação - Contagem e verificação de presença do Timestamp:

"""
Para cada tabela no schema bronze, verifica-se:
- Se a tabela existe (spark.table() não lança exceção)
- Se tem pelo menos 1 registro (count > 0)
- Se a coluna timestamp_ingestion está presente no schema

Caso tenhamos 0 registros, isso indica um problema no caminho do Volume ou no arquivo CSV. Ademais, caso a coluna timestamp_ingestion esteja ausente, isso indica um bug na função.
"""

TABELAS_ESPERADAS = [
    "tb_customers",
    "tb_geolocalizacao",
    "tb_order_items",
    "tb_order_payments",
    "tb_order_reviews",
    "tb_orders",
    "tb_products",
    "tb_sellers",
    "tb_product_category_name_translation",
    "tb_cotacao_dolar",
]

print(f"{'Tabela':<48} {'Registros':>12}  {'Timestamp':>10}  {'Status':>9}")
print("-" * 85)

for tabela in TABELAS_ESPERADAS:
    try:
        df  = spark.table(f"bronze.{tabela}")
        n   = df.count()
        ts  = "Sim" if "timestamp_ingestion" in df.columns else "NAO"
        st  = "OK" if n > 0 and ts == "Sim" else "VERIFICAR"
        print(f"  bronze.{tabela:<40} {n:>12,}  {ts:>10}  {st:>9}")
    except Exception as e:
        print(f"  bronze.{tabela:<40} {'ERRO':>12}  {'---':>10}  {'FALHOU':>9}")
        print(f"    Detalhe: {str(e)[:80]}")

# 5.3 - Amostragem das principais tabelas para o Pipeline:

"""
Aqui, fazemos a exibição dos 3 primeiros registros de cada uma das tabelas principais do pipeline:
- tb_orders : tabela central — todos os pedidos do e-commerce
- tb_order_items : itens e preços — base de cálculo da receita
- tb_cotacao_dolar : valida a ingestão via API ou fallback

Utiliza-se .limit(n): retorna apenas os n primeiros registros sem acionar
um count() completo. Isso é mais eficiente que .show() para validação

Essa amostragem é feita para validar se os dados estão sendo carregados corretamente.
"""

print("Amostra — bronze.tb_orders (3 registros):")
display(spark.table("bronze.tb_orders").limit(3))

print("\nAmostra — bronze.tb_order_items (3 registros):")
display(spark.table("bronze.tb_order_items").limit(3))

print("\nAmostra — bronze.tb_cotacao_dolar (5 registros):")
display(spark.table("bronze.tb_cotacao_dolar").limit(5))

# 5.4 - Inspeção básica do Schema das tabelas críticas

"""
Basicamente, na camada Bronze, os tipos são inferidos automaticamente e podem estar incorretos (ex: CEP inferido como IntegerType perde o zero à esquerda). Isso é esperado e será corrigido posteriormente.

Usa-se printSchema() para exibir a estrutura inferida pelo Spark:
- Nome de cada coluna
- Tipo de dado (StringType, IntegerType, DoubleType, TimestampType...)
- Nullable: se a coluna aceita valores nulos

Verifica-se tb_orders por ser a tabela central, e tb_cotacao_dolar para confirmar que cotacaoCompra foi inferida como DoubleType.
"""

print("Schema inferido — bronze.tb_orders:")
spark.table("bronze.tb_orders").printSchema()

print("Schema inferido — bronze.tb_cotacao_dolar:")
spark.table("bronze.tb_cotacao_dolar").printSchema()

Tabelas registradas no schema 'bronze':


database,tableName,isTemporary
bronze,tb_cotacao_dolar,false
bronze,tb_customers,false
bronze,tb_geolocalizacao,false
bronze,tb_order_items,false
bronze,tb_order_payments,false
bronze,tb_order_reviews,false
bronze,tb_orders,false
bronze,tb_product_category_name_translation,false
bronze,tb_products,false
bronze,tb_sellers,false



Total encontrado: 10/10 tabelas [OK]
Tabela                                              Registros   Timestamp     Status
-------------------------------------------------------------------------------------
  bronze.tb_customers                                   99,441         Sim         OK
  bronze.tb_geolocalizacao                           1,000,163         Sim         OK
  bronze.tb_order_items                                112,650         Sim         OK
  bronze.tb_order_payments                             103,886         Sim         OK
  bronze.tb_order_reviews                              104,162         Sim         OK
  bronze.tb_orders                                      99,441         Sim         OK
  bronze.tb_products                                    32,951         Sim         OK
  bronze.tb_sellers                                      3,095         Sim         OK
  bronze.tb_product_category_name_translation               71         Sim         OK
  bronze.tb_cotac

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,timestamp_ingestion
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,2026-04-04T00:31:16.207Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z,2026-04-04T00:31:16.207Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z,2026-04-04T00:31:16.207Z



Amostra — bronze.tb_order_items (3 registros):


order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,timestamp_ingestion
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.9,13.29,2026-04-04T00:30:59.977Z
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.9,19.93,2026-04-04T00:30:59.977Z
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.0,17.87,2026-04-04T00:30:59.977Z



Amostra — bronze.tb_cotacao_dolar (5 registros):


cotacaoCompra,dataHoraCotacao,timestamp_ingestion
4.038,2016-01-04 13:12:41.021,2026-04-04T00:31:36.557Z
4.0108,2016-01-05 13:12:41.306,2026-04-04T00:31:36.557Z
4.0297,2016-01-06 13:08:04.506,2026-04-04T00:31:36.557Z
4.0469,2016-01-07 13:07:20.817,2026-04-04T00:31:36.557Z
4.0244,2016-01-08 13:11:21.614,2026-04-04T00:31:36.557Z


Schema inferido — bronze.tb_orders:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = true)

Schema inferido — bronze.tb_cotacao_dolar:
root
 |-- cotacaoCompra: double (nullable = true)
 |-- dataHoraCotacao: string (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = true)



In [0]:
""" Consolidação de todas as informações da execução"""

# 6 - Exibição do relatório:
from datetime import datetime

print("Relatório final - Bronze Layer:")
print(f"Concluído em: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")
print(f"Volume utilizado: {VOLUME_PATH}")
print(f"Período PTAX: {DATA_INICIO} → {DATA_FIM}")
print(f"Origem cotação: {'API BCB (online)' if API_ACESSIVEL else 'CSV local (fallback)'}")

print()

print("Tabelas no schema 'bronze':")
for tabela in TABELAS_ESPERADAS:
    try:
        n = spark.table(f"bronze.{tabela}").count()
        print(f"    bronze.{tabela:<43} {n:>10,} registros")
    except:
        print(f"    bronze.{tabela:<43} {'ERRO':>10}")

print()

print("Finalizado!")


Relatório final - Bronze Layer:
Concluído em: 04/04/2026 00:31:56
Volume utilizado: /Volumes/workspace/olist_ecommerce/raw_data
Período PTAX: 01-01-2016 → 12-31-2018
Origem cotação: API BCB (online)

Tabelas no schema 'bronze':
    bronze.tb_customers                                    99,441 registros
    bronze.tb_geolocalizacao                            1,000,163 registros
    bronze.tb_order_items                                 112,650 registros
    bronze.tb_order_payments                              103,886 registros
    bronze.tb_order_reviews                               104,162 registros
    bronze.tb_orders                                       99,441 registros
    bronze.tb_products                                     32,951 registros
    bronze.tb_sellers                                       3,095 registros
    bronze.tb_product_category_name_translation                71 registros
    bronze.tb_cotacao_dolar                                   750 registros

Finalizado!